In [ ]:
%pip install -q kagglehub ultralytics scikit-learn matplotlib seaborn pandas tqdm
%pip install -q opencv-python-headless

import os, shutil, yaml, torch, kagglehub
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

path_ppe = kagglehub.dataset_download("shlokraval/ppe-dataset-yolov8")
print(f"Dataset: {path_ppe}")

WORK_DIR = '/content/PPE_Experiment'
os.makedirs(WORK_DIR, exist_ok=True)

def prepare_dataset(base_path, output_dir, fraction=1.0):
    dataset_dir = os.path.join(output_dir, 'PPE_Data')
    if os.path.exists(dataset_dir):
        shutil.rmtree(dataset_dir)

    for split in ['train', 'valid']:
        for sub in ['images', 'labels']:
            os.makedirs(os.path.join(dataset_dir, split, sub), exist_ok=True)

    for split in ['train', 'valid']:
        src_img = os.path.join(base_path, split, 'images')
        src_lbl = os.path.join(base_path, split, 'labels')
        dst_img = os.path.join(dataset_dir, split, 'images')
        dst_lbl = os.path.join(dataset_dir, split, 'labels')

        if os.path.exists(src_img):
            files = os.listdir(src_img)[:int(len(os.listdir(src_img)) * fraction)]
            for f in files:
                shutil.copy2(os.path.join(src_img, f), os.path.join(dst_img, f))
                lbl_f = f.rsplit('.', 1)[0] + '.txt'
                if os.path.exists(os.path.join(src_lbl, lbl_f)):
                    shutil.copy2(os.path.join(src_lbl, lbl_f), os.path.join(dst_lbl, lbl_f))

    with open(os.path.join(base_path, 'data.yaml'), 'r') as f:
        config = yaml.safe_load(f)

    data_yaml = {
        'train': os.path.abspath(os.path.join(dataset_dir, 'train', 'images')),
        'val': os.path.abspath(os.path.join(dataset_dir, 'valid', 'images')),
        'nc': config['nc'],
        'names': config['names']
    }

    with open(os.path.join(dataset_dir, 'data.yaml'), 'w') as f:
        yaml.dump(data_yaml, f)

    return {
        'yaml_path': os.path.join(dataset_dir, 'data.yaml'),
        'names': config['names'],
        'nc': config['nc'],
        'dir': dataset_dir
    }

ppe_info = prepare_dataset(path_ppe, WORK_DIR)

def calculate_accuracy(model):
    if not hasattr(model, 'validator') or model.validator is None:
        return 0.0
    cm = model.validator.confusion_matrix
    if cm is None or not hasattr(cm, 'matrix'):
        return 0.0
    cm_mat = cm.matrix[:-1, :-1]
    total = np.sum(cm_mat)
    return np.trace(cm_mat) / total if total > 0 else 0.0

print("Training YOLOv8n (25 epochs)...")
model = YOLO('yolov8n.pt')

results = model.train(
    data=ppe_info['yaml_path'],
    epochs=25,
    imgsz=320,
    batch=64,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=8,
    augment=False,
    cache=False,
    verbose=False,
    project=WORK_DIR,
    name='train',
    save=True,
    plots=True,
    close_mosaic=5,
    patience=15
)

print("Validation...")
model.val(data=ppe_info['yaml_path'], verbose=False, plots=False)
accuracy = calculate_accuracy(model)

metrics = results.results_dict.copy()
metrics['metrics/accuracy(B)'] = accuracy

csv_path = os.path.join(results.save_dir, 'results.csv')
df = pd.read_csv(csv_path) if os.path.exists(csv_path) else None

if df is not None:
    df['F1_Score'] = 2 * (df['train/precision'] * df['train/recall']) / (df['train/precision'] + df['train/recall'] + 1e-8)
    df['Total_Loss'] = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']

plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
if df is not None:
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], 'b-', linewidth=2, label='Train', marker='o', markersize=3)
    if 'val/box_loss' in df.columns:
        axes[0, 0].plot(df['epoch'], df['val/box_loss'], 'r--', linewidth=2, label='Val', marker='s', markersize=3)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Box Loss')
    axes[0, 0].set_title('Box Loss Dynamics')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(df['epoch'], df['train/cls_loss'], 'g-', linewidth=2, label='Train', marker='o', markersize=3)
    if 'val/cls_loss' in df.columns:
        axes[0, 1].plot(df['epoch'], df['val/cls_loss'], 'm--', linewidth=2, label='Val', marker='s', markersize=3)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Class Loss')
    axes[0, 1].set_title('Class Loss Dynamics')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(df['epoch'], df['train/precision'], 'b-', linewidth=2, label='Precision', marker='o', markersize=3)
    axes[1, 0].plot(df['epoch'], df['train/recall'], 'r--', linewidth=2, label='Recall', marker='s', markersize=3)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Precision & Recall')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(df['epoch'], df['metrics/mAP50(B)'], 'g-', linewidth=2, label='mAP@0.5', marker='o', markersize=3)
    axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], 'm--', linewidth=2, label='mAP@0.5:0.95', marker='s', markersize=3)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('mAP')
    axes[1, 1].set_title('Mean Average Precision')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'Training_Dynamics.png'), dpi=150, bbox_inches='tight')
plt.show()

cm_paths = [
    os.path.join(results.save_dir, 'confusion_matrix_normalized.png'),
    os.path.join(results.save_dir, 'confusion_matrix.png')
]
for cm_path in cm_paths:
    if os.path.exists(cm_path):
        shutil.copy2(cm_path, os.path.join(WORK_DIR, 'Confusion_Matrix.png'))
        img = plt.imread(cm_path)
        plt.figure(figsize=(14, 12))
        plt.imshow(img)
        plt.title('Confusion Matrix')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(WORK_DIR, 'CM_Display.png'), dpi=150)
        plt.show()
        break

if df is not None:
    corr_cols = ['train/precision', 'train/recall', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
    corr_cols = [c for c in corr_cols if c in df.columns]

    if len(corr_cols) >= 2:
        plt.figure(figsize=(10, 8))
        corr_matrix = df[corr_cols].corr()
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', center=0,
                   square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
        plt.title('Metrics Correlation Matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(WORK_DIR, 'Correlation_Matrix.png'), dpi=150)
        plt.show()

final_metrics = {
    'Metric': ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95', 'Accuracy', 'F1 Score', 'Box Loss', 'Class Loss'],
    'Value': [
        f"{metrics.get('metrics/precision(B)', 0):.4f}",
        f"{metrics.get('metrics/recall(B)', 0):.4f}",
        f"{metrics.get('metrics/mAP50(B)', 0):.4f}",
        f"{metrics.get('metrics/mAP50-95(B)', 0):.4f}",
        f"{metrics.get('metrics/accuracy(B)', 0):.4f}",
        f"{2 * metrics.get('metrics/precision(B)', 0) * metrics.get('metrics/recall(B)', 0) / (metrics.get('metrics/precision(B)', 0) + metrics.get('metrics/recall(B)', 0) + 1e-8):.4f}",
        f"{metrics.get('metrics/box_loss', 0):.4f}",
        f"{metrics.get('metrics/cls_loss', 0):.4f}"
    ]
}

results_df = pd.DataFrame(final_metrics)
print("Final results")
display(results_df)

df.to_csv(os.path.join(WORK_DIR, 'training_history.csv'), index=False) if df is not None else None
results_df.to_csv(os.path.join(WORK_DIR, 'final_metrics.csv'), index=False)

print(f"\n All results saved to: {WORK_DIR}")
print(f" Files:")
for f in os.listdir(WORK_DIR):
    if f.endswith(('.png', '.csv')):
        print(f"   • {f}")